# Chirurgie à chaud — visualisation interactive animée

Notebook **indépendant** (ne dépend d'aucun fichier scratch) : il reconstruit de zéro
l'expérience de chirurgie à chaud (insertion d'un bloc `LlamaBlock` en cours d'entraînement,
via `insert_block!`) et produit un visualiseur HTML interactif autonome
(`graph_surgery_viz.html`, via `save_interactive_graph_animated` de `src/viz.jl`).

Ce que le visualiseur permet :
- un curseur d'époque pour parcourir l'entraînement (avant/après la greffe) ;
- un graphique de perte synchronisé avec le curseur ;
- cliquer sur n'importe quel nœud (paramètre ou activation) pour ouvrir un tableau de ses
  valeurs, avec courbe d'évolution par cellule pour les paramètres ;
- pour les paramètres créés par la greffe (absents avant l'insertion), le panneau affiche
  explicitement **« n'existait pas encore à cette époque »** aux époques antérieures à la
  greffe, au lieu de retomber silencieusement sur leur valeur finale — correctif vérifié dans
  `src/viz.jl` (`fwdContentHTML`) juste avant ce notebook.

In [1]:
using NeuroDSL, Random, Printf

dev = NeuroDSL.Backend.CPUDevice()
ns = :surgery_viz
vocab_size, dim, n_heads, hidden_dim, n_layers, prefix_len = 20, 64, 4, 128, 3, 8

Random.seed!(11)
g, logits = NeuroDSL.build_induction_graph(dev, ns; vocab_size=vocab_size, dim=dim, n_heads=n_heads,
                                            hidden_dim=hidden_dim, n_layers=n_layers, prefix_len=prefix_len)
println("Graphe construit : ", length(g.nodes[ns]), " nœuds (avant greffe)")

Graphe construit : 162 nœuds (avant greffe)


## Boucle d'entraînement instrumentée

Même patron AdamW pas-à-pas que le reste de la session (`m1`/`m2` par symbole de paramètre),
avec capture d'un `TrainingSnapshot` toutes les 25 étapes (`capture_snapshot`) — ces snapshots
sont ce que le visualiseur anime.

In [2]:
ps = NeuroDSL.params(g; namespace=ns)
m1 = Dict(p.name => zeros(Float32, size(p.value)...) for p in ps)
m2 = Dict(p.name => zeros(Float32, size(p.value)...) for p in ps)
rng = MersenneTwister(123)

snapshots = NeuroDSL.TrainingSnapshot[]
all_losses = Float32[]
t = Ref(0)
snapshot_every = 25

function train_step!(param_list; capture=false)
    t[] += 1
    tokens, labels = NeuroDSL.sample_induction_sequence(rng, vocab_size, prefix_len)
    NeuroDSL.set!(g, :token_ids, tokens; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.set!(g, :labels, labels; atom_type=NeuroDSL.Datom, namespace=ns)
    NeuroDSL.invalidate_all!(g; namespace=ns)
    log = NeuroDSL.ExecutionLog()
    loss_val = NeuroDSL.demand!(g, :loss; namespace=ns, log=log)
    NeuroDSL.backward_graph!(g, :loss; namespace=ns, log=log)
    loss_scalar = Float32(sum(Array(loss_val)))
    push!(all_losses, loss_scalar)
    # Capturé ICI, entre backward_graph! et adamw_step! -- adamw_step! remet le
    # gradient à zéro comme dernière étape (voir docstring de capture_snapshot),
    # donc capturer après aurait toujours donné une courbe de gradient nulle.
    if capture
        snap = NeuroDSL.capture_snapshot(g, ns, t[], 1, loss_scalar, log)
        push!(snapshots, snap)
    end
    for p in param_list
        p.gradient === nothing && continue
        NeuroDSL.adamw_step!(dev, p.value, p.gradient, m1[p.name], m2[p.name],
                              3f-3, 0.9f0, 0.999f0, 1f-8, t[], 1f0, 0f0)
    end
    NeuroDSL.invalidate_all!(g; namespace=ns)
end

for step in 1:150
    train_step!(ps; capture=(step % snapshot_every == 0))
end
@printf("Avant greffe : perte moyenne (dernières 10 étapes) = %.4f\n", sum(all_losses[end-9:end]) / 10)

Avant greffe : perte moyenne (dernières 10 étapes) = 3.0807


## Chirurgie à chaud : insertion du bloc

`insert_block!` insère un nouveau `LlamaBlock` après `layer_2_out`, initialisé à l'identité
exacte (projections de sortie MHA/MLP à zéro), et rebranche les consommateurs préexistants —
sans jamais reconstruire le graphe ni perdre l'état de l'optimiseur des paramètres existants.

In [3]:
n_before = length(g.nodes[ns])

# Capturé AVANT insert_block! : insert_block! réécrit la règle des consommateurs
# de :layer_2_out pour qu'ils lisent le nouveau bloc à la place -- cette arête
# d'origine ne survivra dans aucune structure interrogeable après l'appel.
# Conservée ici pour que le visualiseur puisse la redessiner aux époques où
# elle était encore réellement la topologie active (voir graft_edges plus bas).
old_consumers = NeuroDSL.consumers(g, :layer_2_out; ns=ns)
graft_epoch = 150  # dernière époque captée où l'ancienne topologie est valide

# Contrefactuel «sans greffe» -- snapshot de l'état pré-greffe (poids, état
# AdamW, RNG) pour rejouer une continuation SANS insert_block! avec les
# mêmes batches et le même point de départ (voir la cellule après l'entraînement
# post-greffe). copy(rng) fige l'état interne du générateur : rng_ctrl et rng
# produiront ensuite exactement la même séquence de tirages, indépendamment.
pre_graft_params = Dict(p.name => copy(p.value) for p in NeuroDSL.params(g; namespace=ns))
pre_graft_m1 = Dict(k => copy(v) for (k, v) in m1)
pre_graft_m2 = Dict(k => copy(v) for (k, v) in m2)
rng_ctrl = copy(rng)

new_out = NeuroDSL.insert_block!(g, ns, :layer_2_out, dim, n_heads, hidden_dim)
n_after = length(g.nodes[ns])
println("Avant greffe : n_nodes=", n_before)
println("Après greffe : n_nodes=", n_after, "  (+", n_after - n_before, " nœuds)")
println("Consommateurs d'origine de layer_2_out (arêtes remplacées par la greffe) : ", old_consumers)

graft_edges = [(graft_epoch, :layer_2_out, c) for c in old_consumers]

new_ps = [p for p in NeuroDSL.params(g; namespace=ns) if !haskey(m1, p.name)]
println(length(new_ps), " nouveaux paramètres créés par la greffe")
for p in new_ps
    m1[p.name] = zeros(Float32, size(p.value)...)
    m2[p.name] = zeros(Float32, size(p.value)...)
end
NeuroDSL.invalidate_all!(g; namespace=ns)
ps2 = NeuroDSL.params(g; namespace=ns);

Avant greffe : n_nodes=162
Après greffe : n_nodes=212  (+50 nœuds)
Consommateurs d'origine de layer_2_out (arêtes remplacées par la greffe) : [:layer_3_norm1_out, :layer_3_res1]
9 nouveaux paramètres créés par la greffe


In [4]:
for step in 151:300
    train_step!(ps2; capture=(step % snapshot_every == 0))
end
@printf("Après greffe : perte moyenne (dernières 10 étapes) = %.4f\n", sum(all_losses[end-9:end]) / 10)
println("n_snapshots: ", length(snapshots), "  n_losses: ", length(all_losses))

Après greffe : perte moyenne (dernières 10 étapes) = 1.5488
n_snapshots: 12  n_losses: 300


## Contrefactuel : entraînement sans greffe

Même poids et état d'optimiseur qu'au moment de la greffe, mêmes batches (RNG copié), mais **sans** `insert_block!` -- sert de référence pour isoler l'effet de la greffe elle-même sur la courbe de perte, affichée dans le visualiseur via les onglets «With graft» / «Without graft» / «Both».

In [5]:
# Entraînement contrefactuel «sans greffe» : même architecture pré-greffe,
# mêmes poids/état d'optimiseur au point de bascule (pre_graft_*), mêmes
# batches (rng_ctrl, copie de rng au même point) -- isole l'effet de la
# greffe elle-même plutôt que celui du hasard des batches ou d'une
# ré-initialisation des poids.
ns_ctrl = :surgery_viz_ctrl
g_ctrl, _ = NeuroDSL.build_induction_graph(dev, ns_ctrl; vocab_size=vocab_size, dim=dim, n_heads=n_heads,
                                            hidden_dim=hidden_dim, n_layers=n_layers, prefix_len=prefix_len)
for p in NeuroDSL.params(g_ctrl; namespace=ns_ctrl)
    NeuroDSL.set!(g_ctrl, p.name, copy(pre_graft_params[p.name]); is_param=true, namespace=ns_ctrl)
end
m1_ctrl = Dict(k => copy(v) for (k, v) in pre_graft_m1)
m2_ctrl = Dict(k => copy(v) for (k, v) in pre_graft_m2)
ps_ctrl = NeuroDSL.params(g_ctrl; namespace=ns_ctrl)

alt_losses = Float32[]
t_ctrl = Ref(150)
for step in 151:300
    t_ctrl[] += 1
    tokens, labels = NeuroDSL.sample_induction_sequence(rng_ctrl, vocab_size, prefix_len)
    NeuroDSL.set!(g_ctrl, :token_ids, tokens; atom_type=NeuroDSL.Datom, namespace=ns_ctrl)
    NeuroDSL.set!(g_ctrl, :labels, labels; atom_type=NeuroDSL.Datom, namespace=ns_ctrl)
    NeuroDSL.invalidate_all!(g_ctrl; namespace=ns_ctrl)
    loss_val = NeuroDSL.demand!(g_ctrl, :loss; namespace=ns_ctrl)
    NeuroDSL.backward_graph!(g_ctrl, :loss; namespace=ns_ctrl)
    push!(alt_losses, Float32(sum(Array(loss_val))))
    for p in ps_ctrl
        p.gradient === nothing && continue
        NeuroDSL.adamw_step!(dev, p.value, p.gradient, m1_ctrl[p.name], m2_ctrl[p.name],
                              3f-3, 0.9f0, 0.999f0, 1f-8, t_ctrl[], 1f0, 0f0)
    end
    NeuroDSL.invalidate_all!(g_ctrl; namespace=ns_ctrl)
end
@printf("Sans greffe (contrefactuel) : perte moyenne (dernières 10 étapes) = %.4f\n", sum(alt_losses[end-9:end]) / 10)
println("length(alt_losses) = ", length(alt_losses), "  (attendu : ", 300 - graft_epoch, ")")

Sans greffe (contrefactuel) : perte moyenne (dernières 10 étapes) = 1.4127
length(alt_losses) = 150  (attendu : 150)


## Export du visualiseur interactif

`save_interactive_graph_animated` capture la structure du graphe **une seule fois**, à son état
final (post-greffe), et l'anime avec les snapshots capturés pendant l'entraînement. Le fichier
produit est autonome (HTML+JS+CSS inline, aucune dépendance externe) — ouvrable directement
dans un navigateur, sans serveur.

In [6]:
out_path = joinpath(@__DIR__, "graph_surgery_viz.html")
NeuroDSL.save_interactive_graph_animated(g, snapshots, out_path;
                                          title="Hot Graph Surgery -- Live Block Insertion During Training",
                                          losses=all_losses,
                                          graft_edges=graft_edges,
                                          alt_losses=alt_losses)
println("Visualiseur écrit : ", out_path)

✅ Viewer animé exporté → C:\Users\Nevermind\Desktop\NeuroDSL\notebook\graph_surgery_viz.html  (12 snapshots)
Visualiseur écrit : C:\Users\Nevermind\Desktop\NeuroDSL\notebook\graph_surgery_viz.html


## Comment l'utiliser

Ouvrir `graph_surgery_viz.html` dans un navigateur. Le diagramme du graphe lui-même est
désormais dynamique par époque, pas seulement les valeurs :

- **Avant l'époque 150** : le bloc greffé (préfixe `surgery_layer_2_out_...`) est
  entièrement invisible (nœuds ET arêtes) — il n'existait pas encore — et l'arête d'origine
  `layer_2_out → layer_3_...` (remplacée par la greffe) est dessinée à sa place, puisqu'elle
  était réellement la topologie active à ce moment.
- **À partir de l'époque 175** (première capture après la greffe, faite à l'étape 150) : le
  bloc greffé apparaît avec toutes ses arêtes, et l'arête d'origine disparaît.

Cliquer sur un nœud du nouveau bloc à une époque antérieure à la greffe n'est plus possible
depuis le graphe (il est caché) ; on peut toujours vérifier via le panneau qu'il affiche
« n'existait pas encore à cette époque » s'il est ouvert avant que le graphe ne se
réactualise.